# Recover checkpoint predictions and evaluate

Uses the fourteen historical checkpoints if available. It performs inference and metric calculation only.

Run the single code cell below with the **Python (skin)** kernel.

In [ ]:
from pathlib import Path
import sys

PACKAGE_DIR_INPUT = Path(r"D:\AIUB\Neural\A_Comprehensive_Framework_for_Auditing_Demographic_Bias_in_Synthetic_Skin_Disease_Datasets\revision_package")
if not PACKAGE_DIR_INPUT.is_dir():
    raise FileNotFoundError(f"revision_package not found: {PACKAGE_DIR_INPUT}")
if str(PACKAGE_DIR_INPUT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR_INPUT))

import json
import pandas as pd

from notebook_config import (
    CONCAT_CHECKPOINT_DIR,
    DATA_ROOT,
    DSAF_CHECKPOINT_DIR,
    HF_MODEL_SOURCE,
    RECOVERY_PERMUTATIONS,
    REPO_DIR,
    RUN_ROOT,
)
from notebook_support import (
    ensure_core_outputs,
    find_checkpoint_dir,
    run_once,
    select_metadata,
)

metadata_path = select_metadata(require_known_hash=True)
audit_metadata, manifest = ensure_core_outputs(RUN_ROOT)
concat_checkpoints = find_checkpoint_dir(CONCAT_CHECKPOINT_DIR, "concat")
dsaf_checkpoints = find_checkpoint_dir(DSAF_CHECKPOINT_DIR, "dsaf")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 32 if device == "cuda" else 8
print(f"Inference device: {device}; batch size: {batch_size}")

common = [
    "--repo", str(REPO_DIR),
    "--metadata", str(metadata_path),
    "--data-root", str(DATA_ROOT),
    "--backbone-config", HF_MODEL_SOURCE,
    "--confirm-original-row-order",
    "--permutations", str(RECOVERY_PERMUTATIONS),
    "--batch-size", str(batch_size),
    "--device", device,
]

concat_dir = run_once(
    "04_recover_oof.py",
    [*common, "--kind", "concat", "--checkpoint-dir", str(concat_checkpoints), "--save-features"],
    RUN_ROOT / "recovered_concat",
    ["oof_predictions.csv", "classes.json", "reconstructed_fold_manifest.csv", "frozen_cls_features.npz"],
)
dsaf_dir = run_once(
    "04_recover_oof.py",
    [*common, "--kind", "dsaf", "--checkpoint-dir", str(dsaf_checkpoints)],
    RUN_ROOT / "recovered_dsaf",
    ["oof_predictions.csv", "classes.json", "reconstructed_fold_manifest.csv"],
)

concat_classes = json.loads((concat_dir / "classes.json").read_text(encoding="utf-8"))
dsaf_classes = json.loads((dsaf_dir / "classes.json").read_text(encoding="utf-8"))
if concat_classes != dsaf_classes:
    raise ValueError("Recovered class orders differ; do not combine predictions")

evaluation = run_once(
    "03_evaluate_predictions.py",
    [
        "--predictions", str(concat_dir / "oof_predictions.csv"), str(dsaf_dir / "oof_predictions.csv"),
        "--classes", str(concat_dir / "classes.json"),
        "--metadata", str(audit_metadata),
        "--manifest", str(manifest),
        "--compare", "concat,dsaf",
    ],
    RUN_ROOT / "prediction_audit",
    ["overall.csv", "disease_metrics.csv", "support_sensitivity.csv", "eod_per_class.csv", "paired_comparison.json"],
)

print("\nRecovered held-out results:")
print(pd.read_csv(evaluation / "overall.csv").to_string(index=False))
print("\nPaired comparison:")
print((evaluation / "paired_comparison.json").read_text(encoding="utf-8"))
print("\nNo model training occurred. Recovery stops automatically if historical fold accuracies do not match.")
